In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.rcParams["font.family"] = "monospace"


In [ ]:
with open("../data/models_members.json", "r") as f:
    models_members = json.load(f)

def _read_group(name):
    with open(f"../data/{name}_models.txt") as f:
        return [line.strip() for line in f if line.strip()]

green_models = _read_group("green")
orange_models = _read_group("orange")
red_models = _read_group("red")


In [ ]:
def create_metrics_df(metric_type, metrics_dir, models_members):
    """
    Loads and processes metrics data from JSON files for a specific metric type.

    Args:
        metric_type (str): The type of metric (e.g., 'proc', 'perf', 'tel').
        metrics_dir (str): The directory path containing the JSON files.
        models_members (dict): Dictionary of models and their members to process.

    Returns:
        pd.DataFrame: A DataFrame containing the processed metrics.
    """
    all_metrics_data = []

    for model, _members in models_members.items():
        for member in _members:
            filename = f"cmip6_historical_ENSO_{metric_type}_EnsoCMIP6regrid_{model}_{member}.json"
            metrics_path = os.path.join(metrics_dir, filename)

            if not os.path.exists(metrics_path):
                print(f"Warning: File not found, skipping. {metrics_path}")
                continue

            with open(metrics_path) as f:
                _metrics_data = json.load(f)

            metrics_data = _metrics_data["RESULTS"]["model"][model][member]["value"]
            _row_value = {"model": model, "member": member}

            for metric, data in metrics_data.items():
                _value = pd.DataFrame(data["metric"]).mean(axis=1)["value"]
                _row_value[metric] = _value

            all_metrics_data.append(_row_value)

    final_df = pd.DataFrame(all_metrics_data)
    if not final_df.empty:
        final_df = final_df.set_index(["model", "member"])

    return final_df


## PMP ENSO metrics + coastal statistics

In [ ]:
proc_metrics_dir = "../data/EnsoCMIP6regrid/ENSO_proc/"
perf_metrics_dir = "../data/EnsoCMIP6regrid/ENSO_perf/"
tel_metrics_dir = "../data/EnsoCMIP6regrid/ENSO_tel/"

proc_metrics_df = create_metrics_df("proc", proc_metrics_dir, models_members)
perf_metrics_df = create_metrics_df("perf", perf_metrics_dir, models_members)
tel_metrics_df = create_metrics_df("tel", tel_metrics_dir, models_members)

metrics_raw = (
    proc_metrics_df
    .merge(perf_metrics_df, on=["model", "member"], suffixes=("", "_drop"))
    .merge(tel_metrics_df, on=["model", "member"], suffixes=("", "_drop"))
)
metrics_raw = metrics_raw.drop(
    columns=[c for c in metrics_raw.columns if c.endswith("_drop")]
)
metrics = metrics_raw.groupby("model").mean()


In [ ]:
coastal_stats = pd.read_csv("../data/coastal_stats_esgf.csv")

# Mean of the two obs rows (always appended last) as the observed reference
obs_rows = coastal_stats.iloc[-2:]
numeric_cols = coastal_stats.select_dtypes(include="number").columns
obs_ref = obs_rows[numeric_cols].mean().to_dict()

coastal_stats = coastal_stats.iloc[:-2, :]  # drop obs rows
coastal_stats_by_model = (
    coastal_stats.groupby("model")[coastal_stats.columns[2:]].mean().reset_index()
)


In [ ]:
perf_models = metrics.reset_index()
common_models = list(
    set(perf_models["model"]).intersection(set(coastal_stats_by_model["model"]))
)

common_perf_df = perf_models.query("model in @common_models").copy()
common_coa_df = coastal_stats_by_model.query("model in @common_models").copy()

for col in common_coa_df.columns[1:]:
    common_perf_df[col] = common_coa_df[col].values
common_perf_df = common_perf_df.dropna(axis=1)

df = common_perf_df.copy()


def _assign_group(model):
    if model in green_models:
        return "green"
    if model in orange_models:
        return "orange"
    if model in red_models:
        return "red"
    return "other"


df["group"] = df["model"].apply(_assign_group)
df.head()


## Feedback slopes

In [ ]:
FEEDBACKS = ["EnsoFbSstTaux", "EnsoFbTauxSsh", "EnsoFbSshSst", "EnsoFbSstThf"]


def load_feedback_slopes(models_members, proc_dir):
    """Return (per-model slope DataFrame, obs-slope dict) for the feedbacks."""
    rows = []
    obs_acc = {fb: [] for fb in FEEDBACKS}
    for model, members in models_members.items():
        for member in members:
            f = os.path.join(
                proc_dir,
                f"cmip6_historical_ENSO_proc_EnsoCMIP6regrid_{model}_{member}.json",
            )
            if not os.path.exists(f):
                continue
            with open(f) as fh:
                d = json.load(fh)
            v = d["RESULTS"]["model"][model][member]["value"]
            mkey = f"{model}_{member}"
            row = {"model": model, "member": member}
            for fb in FEEDBACKS:
                diag = v.get(fb, {}).get("diagnostic", {})
                if not diag:
                    continue
                if isinstance(diag.get(mkey), dict) and diag[mkey].get("value") is not None:
                    row[fb] = diag[mkey]["value"]
                for k, val in diag.items():
                    if k != mkey and isinstance(val, dict) and val.get("value") is not None:
                        obs_acc[fb].append(val["value"])
            rows.append(row)
    slopes_df = pd.DataFrame(rows)
    slopes = (
        slopes_df.drop(columns=["member"]).groupby("model").mean(numeric_only=True)
        if not slopes_df.empty
        else pd.DataFrame()
    )
    obs = {
        fb: (float(np.nanmean(obs_acc[fb])) if obs_acc[fb] else np.nan)
        for fb in FEEDBACKS
    }
    return slopes, obs


slopes, obs_slopes = load_feedback_slopes(models_members, proc_metrics_dir)
for fb in FEEDBACKS:
    if fb in slopes.columns:
        df[fb] = df["model"].map(slopes[fb])
        obs_ref[fb] = obs_slopes[fb]


## Figure

In [ ]:
LABELS = {
    "EnsoSstLonRmse": "ENSO pattern", "BiasSstLonRmse": "Zonal SST (bias)",
    "BiasPrLonRmse": "Zonal PR (bias)", "BiasPrLatRmse": "Meridional PR/ITCZ (bias)",
    "BiasTauxLonRmse": r"Zonal $\tau_x$ (bias)",
    "SeasonalSstLonRmse": "Zonal SST (seasonal)", "SeasonalPrLonRmse": "Zonal PR (seasonal)",
    "SeasonalPrLatRmse": "Meridional PR/ITCZ (seasonal)",
    "SeasonalTauxLonRmse": r"Zonal $\tau_x$ (seasonal)",
    "EnsoFbSstTaux": r"SST-$\tau_x$ feedback",
    "EnsoFbTauxSsh": r"$\tau_x$-SSH feedback", "EnsoFbSshSst": "SSH-SST feedback",
    "alpha": r"$\alpha$-value", "en34_std": "EN34 SST std",
    "en12_pr_std": "EN12 PR std", "en12_std": "EN12 SST std",
    "EnsoAmpl": "ENSO amplitude",
    "standalone_coa_per_century": "Standalone COA frequency",
    "nonstandalone_coa_per_century": "Spreading COA frequency",
}
IS_ERR = {"EnsoSstLonRmse", "BiasSstLonRmse", "BiasPrLonRmse", "BiasPrLatRmse",
          "BiasTauxLonRmse", "SeasonalSstLonRmse", "SeasonalPrLonRmse",
          "SeasonalPrLatRmse", "SeasonalTauxLonRmse"}
UNITS = {  # PMP feedback diagnostic units (regression slopes)
    "EnsoFbSstTaux": r"$10^{-3}$ N m$^{-2}$ $^\circ$C$^{-1}$",
    "EnsoFbTauxSsh": r"$10^{3}$ cm (N m$^{-2})^{-1}$",
    "EnsoFbSshSst": r"$^\circ$C cm$^{-1}$",
    "en34_std": r"$^\circ$C",
    "standalone_coa_per_century": r"century$^{-1}$",
    "nonstandalone_coa_per_century": r"century$^{-1}$",
}
BIAS_METRICS = ["BiasSstLonRmse", "SeasonalSstLonRmse", "BiasPrLonRmse",
                "SeasonalPrLonRmse", "BiasPrLatRmse", "SeasonalPrLatRmse",
                "BiasTauxLonRmse", "SeasonalTauxLonRmse"]
PROP_METRICS = ["EnsoSstLonRmse", "EnsoAmpl", "alpha", "en12_std", "en34_std",
                "en12_pr_std"]
# per-group feedback <-> COA-frequency scatter panels
SCATTER_PANELS = [("EnsoFbSstTaux", "standalone_coa_per_century"),
                  ("EnsoFbTauxSsh", "standalone_coa_per_century"),
                  ("EnsoFbTauxSsh", "nonstandalone_coa_per_century"),
                  ("EnsoFbSshSst", "standalone_coa_per_century")]
TARGETS = [("coa_per_century", "Total", "#333333"),
           ("standalone_coa_per_century", "Standalone", "#1f77b4"),
           ("nonstandalone_coa_per_century", "Spreading", "#d62728")]
OFF = {"Total": 0.26, "Standalone": 0.0, "Spreading": -0.26}
GROUP_PALETTE = {"orange": "#FFA726", "green": "#2E7D32", "red": "#D32F2F"}
GROUP_DISPLAY = {"orange": "Underestimators", "green": "Good COA", "red": "Overestimators"}


In [ ]:
def _forest(ax, df, metrics, title, legend=False):
    n = len(metrics)
    for i, col in enumerate(metrics):
        ypos = n - 1 - i
        for tcol, short, c in TARGETS:
            s = df[[col, tcol]].dropna()
            if len(s) <= 3:
                continue
            r, p = stats.pearsonr(s[col], s[tcol])
            y = ypos + OFF[short]
            ax.plot([0, r], [y, y], color=c, lw=1.3, zorder=1)
            ax.plot(r, y, "o", ms=6, zorder=2, mfc=c if p < 0.05 else "white", mec=c)
    for k in range(n - 1):  # separators between metric groups
        ax.axhline(k + 0.5, color="grey", lw=0.6, ls="--", alpha=0.5, zorder=0)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_yticks(range(n))
    ax.set_yticklabels([LABELS.get(m, m) for m in metrics][::-1], fontsize=9)
    ax.set_ylim(-0.6, n - 0.4)
    ax.set_xlim(-0.7, 0.7)
    ax.set_xlabel("Pearson r with COA frequency", fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.grid(axis="x", alpha=0.3, ls="--")
    if legend:
        h = [plt.Line2D([], [], marker="o", ls="", mfc=c, mec=c, label=short)
             for _, short, c in TARGETS]
        ax.legend(handles=h, loc="upper left", fontsize=8.5)


def _lab(c):
    return LABELS[c] + (f" [{UNITS[c]}]" if c in UNITS else "")


def _scatter_grouped(ax, xc, yc, lett):
    """Per-group scatter: one regression line and one Pearson r per model
    group (no all-collection fit)."""
    s = df[[xc, yc, "group"]].dropna()
    rows = []
    for i, g in enumerate(["orange", "green", "red"]):
        gdf = s[s["group"] == g]
        col = GROUP_PALETTE.get(g, "0.5")
        ax.scatter(gdf[xc], gdf[yc], s=34, color=col, edgecolor="k",
                   linewidth=0.3, zorder=3)
        if len(gdf) >= 4:
            r, p = stats.pearsonr(gdf[xc], gdf[yc])
            sns.regplot(data=gdf, x=xc, y=yc, ax=ax, scatter=False, color=col,
                        line_kws=dict(lw=1.7), ci=95, truncate=True)
            ax.text(0.03, 0.965 - 0.05 * len(rows),
                    f"r={r:+.2f}{'*' if p < .05 else ''}",
                    transform=ax.transAxes, ha="left", va="top", fontsize=8.5,
                    color=col, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white",
                              ec="none", alpha=0.75))
            rows.append((g, r, p))
    ox = 0 if xc in IS_ERR else obs_ref.get(xc, np.nan)
    oy = 0 if yc in IS_ERR else obs_ref.get(yc, np.nan)
    if np.isfinite(ox) and np.isfinite(oy):
        ax.axvline(ox, color="grey", lw=0.6, ls=":")
        ax.axhline(oy, color="grey", lw=0.6, ls=":")
        ax.scatter(ox, oy, marker="D", s=90, c="grey", edgecolors="black",
                   linewidths=0.8, zorder=6)
    ax.text(0.02, 1.02, lett, transform=ax.transAxes, ha="left", va="bottom",
            fontsize=10, fontweight="bold")
    ax.set_xlabel(_lab(xc), fontsize=8.5)
    ax.set_ylabel(_lab(yc), fontsize=8.5)
    ax.grid(alpha=0.25, ls="--")


In [ ]:
legend_handles = [
    plt.Line2D([], [], marker="o", ls="", mfc=GROUP_PALETTE[g], mec="k",
               label=GROUP_DISPLAY[g]) for g in ["orange", "green", "red"]]
legend_handles.append(plt.Line2D([], [], marker="D", ls="", mfc="grey",
                                 mec="black", ms=9, label="Observations"))

fig = plt.figure(figsize=(10.5, 11.8), dpi=300)
gs = fig.add_gridspec(3, 2, height_ratios=[1.15, 1, 1], hspace=0.23, wspace=0.16)
_bias = [m for m in BIAS_METRICS if m in df.columns]
_prop = [m for m in PROP_METRICS if m in df.columns]
_forest(fig.add_subplot(gs[0, 0]), df, _bias,
        "(a) Mean-state and seasonal-cycle biases")
axb = fig.add_subplot(gs[0, 1])
_forest(axb, df, _prop, "(b) ENSO properties", legend=True)
axb.yaxis.tick_right()
axb.yaxis.set_label_position("right")

# bottom rows: per-group feedback <-> COA-frequency scatters
bb = {}
for key, cell, (xc, yc), lett in zip(
        ["c", "d", "e", "f"], [gs[1, 0], gs[1, 1], gs[2, 0], gs[2, 1]],
        SCATTER_PANELS, ["(c)", "(d)", "(e)", "(f)"]):
    bb[key] = fig.add_subplot(cell)
    _scatter_grouped(bb[key], xc, yc, lett)

# standalone-frequency panels (c, d, f) share one y-range
_yl = [bb[k].get_ylim() for k in ("c", "d", "f")]
_ylim = (min(y[0] for y in _yl), max(y[1] for y in _yl))
for k in ("c", "d", "f"):
    bb[k].set_ylim(_ylim)
_y0 = min(bb["e"].get_position().y0, bb["f"].get_position().y0)
fig.legend(handles=legend_handles, loc="upper center", ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, _y0 - 0.04))
